In [1]:
import random
import numpy as np

random.seed(189)
np.random.seed(189)

from sklearn import svm
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from scipy import io
import pandas as pd

In [2]:
spam = np.load("/kaggle/input/spam-data/spam-data.npz") # May have to change file path
spam_training_data = spam['training_data']
spam_training_labels = spam['training_labels']
spam_test_data = spam['test_data']

In [3]:
# Function that shuffles data given a dataset and its labels
def shuffle_data_labels(data, labels):
    data_points = np.arange(data.shape[0])
    np.random.shuffle(data_points)
    return [data[data_points], labels[data_points]]
    
# Shuffle dataset
spam_training_data, spam_training_labels = shuffle_data_labels(spam_training_data, spam_training_labels)

# Partition the shuffled dataset
spam_training_data_len = spam_training_data.shape[0]
spam_training_set_data, spam_training_set_labels = spam_training_data[:int(0.8 * spam_training_data_len), :], spam_training_labels[:int(0.8 * spam_training_data_len)]
spam_validation_set_data, spam_validation_set_labels = spam_training_data[int(0.8 * spam_training_data_len):, :], spam_training_labels[int(0.8 * spam_training_data_len):]

In [4]:
# Standardize the data
scaler = StandardScaler()
spam_training_set_data_scaled = scaler.fit_transform(spam_training_set_data)
spam_validation_set_data_scaled = scaler.transform(spam_validation_set_data)
spam_test_data_scaled = scaler.transform(spam_test_data)

In [5]:
# Perform Grid Search to find best parameters for best model
parameter_grid = {
    'C': [1, 10, 30, 50, 100],
    'kernel': ['linear', 'rbf'],
    'class_weight': ['balanced', None]
}

model = SVC(max_iter=1000000, probability=True)
grid_search = GridSearchCV(model, parameter_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(spam_training_set_data_scaled, spam_training_set_labels)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_}")

best_model = grid_search.best_estimator_

Best parameters: {'C': 1, 'class_weight': None, 'kernel': 'rbf'}
Best cross-validation score: 0.9640292129384409


In [6]:
# Evaluate model
training_accuracy = best_model.score(spam_training_set_data_scaled, spam_training_set_labels)
validation_accuracy = best_model.score(spam_validation_set_data_scaled, spam_validation_set_labels)
print(f"Training Accuracy: {training_accuracy}")
print(f"Validation Accuracy: {validation_accuracy}")

Training Accuracy: 0.9949040767386091
Validation Accuracy: 0.9652694610778443


In [7]:
# Generate CSV file
def results_to_csv(y_test, file_name):
    y_test = y_test.astype(int)
    df = pd.DataFrame({'Category': y_test})
    df.index += 1
    df.to_csv(file_name, index_label='Id')

In [8]:
y_test = best_model.predict(spam_test_data_scaled)
results_to_csv(y_test, 'spam-submission.csv')
print("Submission file 'spam-submission.csv' has been created.")

Submission file 'spam-submission.csv' has been created.
